In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import warnings
warnings.filterwarnings("ignore")

builder = (
    SparkSession.builder
    .appName("delta-minio-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.2.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e6f5ea65-98e4-4f18-831e-8969649553d7;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 128ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [8]:
!spark-submit \
  --packages io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 \
  --conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" \
  --conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" \
  /workspace/rltm_bi_pltfrm/jobs/silver_to_gold/gold_price_metrics.py

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6f3d19d1-909a-4558-8541-053283fc17e2;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 153ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 f

In [9]:
metrics_path = "s3a://lakehouse/gold/gold_price_metrics"

df = spark.read.format("delta").load(metrics_path)
print("row_count =", df.count())
df.orderBy("metric_ts", ascending=False).show(20, truncate=False)

26/03/21 23:42:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


row_count = 9
+------+------------+--------+-------------------+-----------+---------+---------+-----------+-----------+-----------+-------------+-------------+---------+----------+
|symbol|source_name |currency|metric_ts          |price_usd  |return_1m|return_5m|ma_5       |ma_15      |ma_60      |volatility_15|volatility_60|zscore_60|is_anomaly|
+------+------------+--------+-------------------+-----------+---------+---------+-----------+-----------+-----------+-------------+-------------+---------+----------+
|XAU   |gold_api_com|USD     |2026-03-21 17:31:00|4492.200195|0.0      |0.0      |4492.200195|4492.200195|4492.200195|0.0          |0.0          |NULL     |false     |
|XAU   |gold_api_com|USD     |2026-03-21 17:18:00|4492.200195|0.0      |0.0      |4492.200195|4492.200195|4492.200195|0.0          |0.0          |NULL     |false     |
|XAU   |gold_api_com|USD     |2026-03-21 17:17:00|4492.200195|0.0      |0.0      |4492.200195|4492.200195|4492.200195|0.0          |0.0          |